# Final Customer Segmentation Project (Alfido Tech)
Includes Cleaning, Feature Engineering, RFM, KMeans, Visualization, Insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load Data
df = pd.read_csv('ecommerce_customer_data_custom_ratios.csv')
df.head()

In [ ]:
# Data Cleaning
df.drop_duplicates(inplace=True)
df.fillna(method='ffill', inplace=True)
df.info()

In [ ]:
# Feature Engineering
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [ ]:
# RFM Analysis
latest_date = df['InvoiceDate'].max()
rfm = df.groupby('CustomerID').agg({
'InvoiceDate': lambda x: (latest_date - x.max()).days,
'InvoiceNo': 'count',
'TotalAmount': 'sum'})
rfm.columns=['Recency','Frequency','Monetary']
rfm.head()

In [ ]:
# Scaling
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

In [ ]:
# KMeans Clustering
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
# Segment Profiling
segment_profile = rfm.groupby('Cluster').mean()
segment_profile

In [ ]:
# Monthly Sales Trend
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df.groupby('Month')['TotalAmount'].sum().plot()
plt.title('Monthly Sales Trend')
plt.show()

In [ ]:
# Retention Analysis
customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
sns.histplot(customer_orders)
plt.title('Customer Retention Distribution')
plt.show()

In [ ]:
# Correlation Heatmap
sns.heatmap(rfm.corr(), annot=True)
plt.show()